# 2단계 — ResNet18 분류기 학습 (SOC-50)

## 이 노트북의 역할
2단계 파이프라인(탐지 → 분류)의 **뒷단**: 탐지기가 찾은 영역(크롭된 슬라이스)을 받아
**50종 차종 중 무엇인지 판별**한다. 이 노트북은 분류기 후보 5개 중 ResNet18이며,
같은 구조로 모델 정의 셀만 바꿔 5개(SARATR-X, ConvNeXt, ResNet34, ResNet18, HDANet)를 학습 후 비교한다.

## SOC-50 설정이란
- ATRNet-STAR 40종 + MSTAR 10클래스(군용차량) = **50클래스** 분류 문제
- 두 데이터셋은 센서/장면 특성이 다르므로 보고서에 혼합 데이터임을 명시할 것
- **논문(ATRBench) 기준치 (Overall Accuracy)**: SARATR-X 85.2% / ConvNeXt 81.6% /
  ResNet34 72.9% / ResNet18 **71.2%** / HDANet 63.7% ← 우리 결과의 비교 기준

## 데이터 형식 참고 (stage1과의 차이)
stage1(YOLO)에서는 tif가 1채널로 읽혀 PNG 변환이 필요했지만, **이 노트북은 변환 불필요** —
PyTorch의 ImageFolder는 PIL로 이미지를 읽고, `Grayscale(num_output_channels=3)` 변환이
1채널 tif를 3채널로 만들어주기 때문. (stage1의 문제는 ultralytics가 cv2로 tif를 읽는 방식 때문이었음)

## 안전장치
에폭마다 체크포인트(모델+옵티마이저+스케줄러 상태)를 드라이브에 저장 →
세션이 끊겨도 노트북을 처음부터 재실행하면 **자동으로 마지막 에폭부터 이어서 학습**.

## 사전 준비
- 드라이브 `MyDrive/ATRNet-STAR/soc50.tar`
- 런타임 → **T4 GPU** (예상 시간: 20에폭 ~20-30분)

## 1. GPU 확인 + 드라이브 마운트

GPU가 안 잡히면 학습이 수십 배 느려지므로 반드시 먼저 확인.
드라이브는 데이터 읽기 + 체크포인트/결과 저장용.

In [ ]:
import torch

# "Tesla T4"가 출력되어야 정상
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 — 런타임 유형 변경!")

from google.colab import drive
drive.mount('/content/drive')   # 계정 선택 창 — 데이터가 있는 계정 선택

GPU: Tesla T4
Mounted at /content/drive


## 2. 데이터를 코랩 로컬 디스크로 복사

**왜?** 드라이브 마운트에서 작은 파일 수만 개를 직접 읽으면 학습이 3~10배 느려짐.
tar 하나만 복사해 로컬에 해제하는 방식이 가장 빠르다 (수 분).
이미 해제돼 있으면 건너뛰므로 재실행에 안전.

In [ ]:
import os

SETTING     = "SOC_50classes"
TAR_PATH    = "/content/drive/MyDrive/ATRNet-STAR/soc50.tar"      # 빠른 경로
FOLDER_PATH = f"/content/drive/MyDrive/ATRNet-STAR/{SETTING}"     # tar 없을 때 대체 경로
DATA_DIR    = f"/content/{SETTING}"                               # 코랩 로컬 목적지

if not os.path.exists(DATA_DIR):                  # 이미 준비됐으면 건너뜀
    if os.path.exists(TAR_PATH):
        print("tar 발견 → 빠른 경로")
        !cp "{TAR_PATH}" /content/soc50.tar
        !tar -xf /content/soc50.tar -C /content/ && rm /content/soc50.tar
    else:
        print("tar 없음 → 폴더 복사 (느림) 후 다음 세션용 tar 생성")
        !cp -r "{FOLDER_PATH}" /content/
        !tar -cf "{TAR_PATH}" -C /content {SETTING}

!ls "{DATA_DIR}"                                # train/test 폴더명 확인 → 다음 셀에 반영
!find "{DATA_DIR}" -name "*.tif" | wc -l        # 전체 이미지 수 (~35,000 내외)

test  train
35674


## 3. 데이터셋 / 데이터로더 정의

### 각 구성 요소의 의미
- **`ImageFolder`**: "클래스별 하위 폴더" 구조를 자동 인식해 (이미지, 라벨) 쌍 생성.
  라벨 번호는 폴더명 **알파벳순** — 5개 모델 노트북 모두 같은 순서가 되므로 비교 가능.
  xml 등 이미지가 아닌 파일은 자동 무시
- **`Resize((128,128))`**: MSTAR 이미지는 크기가 제각각(54~193px)이라 128로 통일
- **`Grayscale(num_output_channels=3)`**: SAR 흑백 1채널 → 같은 값을 3채널로 복제.
  ImageNet 사전학습 모델(ResNet 등)이 3채널(RGB) 입력을 기대하기 때문
- **`ToTensor()`**: 픽셀값 0~255 → 0~1 변환 + PyTorch 텐서화
- **`batch_size=256`**: T4(16GB)에서 128px 이미지 기준 무리 없음. OOM 에러 시 128로
- **`num_workers=2`**: 데이터 로딩 병렬화 (코랩 CPU 2코어 기준 적정값)
- **`pin_memory=True`**: CPU→GPU 전송 가속
- train은 `shuffle=True`(에폭마다 순서 섞어 일반화), test는 `False`(평가 재현성)

In [ ]:
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# ▼ 셀 2 출력에서 확인한 실제 폴더명으로 수정
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR  = os.path.join(DATA_DIR, "test")

transform = T.Compose([
    T.Resize((128, 128)),                 # MSTAR 크기 불일치 대비 통일
    T.Grayscale(num_output_channels=3),   # 1채널 → 3채널 복제 (ImageNet 모델 호환)
    T.ToTensor(),                          # [0,255] → [0,1] 텐서
])

# train용 (증강 포함) — test는 기존 transform 그대로!
train_transform = T.Compose([
    T.Resize((128, 128)),
    T.Grayscale(num_output_channels=3),
    T.RandomHorizontalFlip(),                          # 좌우 반전
    T.RandomAffine(degrees=10, translate=(0.1, 0.1)),  # 약간의 회전/이동
    T.ToTensor(),
    T.RandomErasing(p=0.3),                            # 일부 영역 가림 (가림/폐색 강인성)
])

train_ds = ImageFolder(TRAIN_DIR, transform=train_transform)
test_ds  = ImageFolder(TEST_DIR,  transform=transform)   # test는 증강 없이

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

NUM_CLASSES = len(train_ds.classes)
print(f"클래스 수: {NUM_CLASSES} (50이어야 정상)")
print(f"train: {len(train_ds)}장 / test: {len(test_ds)}장")
print("클래스 순서(앞 5개):", train_ds.classes[:5])   # 알파벳순 — 모든 모델에서 동일해야 함

클래스 수: 50 (50이어야 정상)
train: 18071장 / test: 17603장
클래스 순서(앞 5개): ['2S1', 'BMP2', 'BRDM_2', 'BTR70', 'BTR_60']


## 4. 모델 정의 — ResNet18 (ImageNet 사전학습)

### 구성 설명
- **`resnet18(weights=DEFAULT)`**: ImageNet 사전학습 가중치 로드 (전이학습).
  처음부터 학습하는 것보다 수렴이 빠르고 최종 정확도도 높음
- **`model.fc = nn.Linear(...)`**: 마지막 분류층을 ImageNet 1000클래스 → 50클래스로 교체.
  이 층만 새로 초기화되고 나머지 가중치는 유지
- **`CrossEntropyLoss`**: 다중 분류 표준 손실 함수
- **`AdamW (lr=1e-4)`**: 파인튜닝용 낮은 학습률 — 사전학습 가중치를 크게 망가뜨리지 않도록
- **`CosineAnnealingLR`**: 학습률을 코사인 곡선으로 서서히 감소 → 후반 안정적 수렴
- **`weight_decay=1e-4`**: 과적합 방지 정규화

### ⚠ 공정 비교 원칙
**하이퍼파라미터(EPOCHS, LR, batch, transform)는 5개 모델 모두 동일하게 유지.**
early stopping을 쓰지 않는 이유: 모델마다 멈추는 에폭이 달라지면 성능 차이가
"모델 능력"인지 "학습량"인지 해석이 흐려짐 + test셋이 학습 중단 결정에 개입하는 정보 누출 문제.

### 다른 모델 노트북 만들 때 (이 셀만 교체)
| 모델 | 생성 코드 | 분류층 교체 위치 |
|------|----------|----------------|
| ResNet34 | `resnet34(weights='DEFAULT')` | `model.fc` |
| ConvNeXt | `convnext_tiny(weights='DEFAULT')` | `model.classifier[2]` |
| HDANet / SARATR-X | 공식 GitHub repo 코드 | repo 구조 따름 |

`MODEL_NAME`도 함께 변경 (체크포인트/결과 파일명에 사용됨).

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# ===== 실험 설정 (5개 모델 공통 — 절대 모델마다 바꾸지 말 것) =====
MODEL_NAME = "resnet18_soc50"     # 파일명에 사용 — 모델마다 변경
EPOCHS = 30
LR = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ImageNet 사전학습 가중치로 모델 생성 후 분류층만 50클래스로 교체
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # 과신 억제
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)    # 옵티마이저
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS) # 학습률 스케줄

# 드라이브 저장 폴더 준비 (체크포인트 + 결과 json)
SAVE_DIR = "/content/drive/MyDrive/ATRNet-STAR"
os.makedirs(f"{SAVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{SAVE_DIR}/results", exist_ok=True)
CKPT_PATH = f"{SAVE_DIR}/checkpoints/{MODEL_NAME}_ckpt.pth"

# 파라미터 수 — 모델 비교표에 사용 (ResNet18 ≈ 11.2M)
print(f"{MODEL_NAME} 파라미터: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 141MB/s]


resnet18_soc50 파라미터: 11.2M


## 5. 학습 루프

### 구조
에폭마다: [train 전체 1회 순회 → 학습률 갱신 → 드라이브에 체크포인트 저장]

### 한 스텝의 흐름 (안쪽 for문)
1. `optimizer.zero_grad()` — 이전 배치의 기울기 초기화
2. `model(x)` — 순전파: 현재 가중치로 예측
3. `criterion(out, y)` — 예측과 정답의 차이(손실) 계산
4. `loss.backward()` — 역전파: 각 가중치의 기울기 계산
5. `optimizer.step()` — 기울기 방향으로 가중치 갱신

### 체크포인트 이어 학습 원리
시작 시 드라이브에 체크포인트가 있으면 모델/옵티마이저/스케줄러 상태와 에폭 번호를 복원해
`start_epoch`부터 재개 → 세션이 끊겨도 손실 없음. 처음부터 다시 학습하려면
드라이브의 `{MODEL_NAME}_ckpt.pth`를 삭제하고 실행.

In [ ]:
import time

# ----- 체크포인트가 있으면 이어서 학습 -----
start_epoch, history = 0, []
if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])             # 모델 가중치 복원
    optimizer.load_state_dict(ckpt["optimizer"])     # 옵티마이저 상태(모멘텀 등) 복원
    scheduler.load_state_dict(ckpt["scheduler"])     # 학습률 스케줄 위치 복원
    start_epoch, history = ckpt["epoch"] + 1, ckpt["history"]
    print(f"체크포인트 발견 → 에폭 {start_epoch + 1}부터 재개")

t0 = time.time()
for epoch in range(start_epoch, EPOCHS):
    model.train()                          # 학습 모드 (BatchNorm/Dropout 활성)
    total_loss, correct, n = 0.0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()              # 1) 기울기 초기화
        out = model(x)                     # 2) 순전파
        loss = criterion(out, y)           # 3) 손실 계산
        loss.backward()                    # 4) 역전파
        optimizer.step()                   # 5) 가중치 갱신

        # 에폭 통계 누적
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        n += x.size(0)

    scheduler.step()                       # 에폭 단위 학습률 감소

    history.append({"epoch": epoch, "train_loss": total_loss/n, "train_acc": correct/n})
    print(f"Epoch {epoch+1}/{EPOCHS} | loss {total_loss/n:.4f} | acc {correct/n:.4f} "
          f"| 누적 {(time.time()-t0)/60:.1f}분")

    # ----- 매 에폭 드라이브에 체크포인트 저장 (세션 끊김 대비) -----
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "epoch": epoch, "history": history},
               CKPT_PATH)

print("학습 완료")

Epoch 1/30 | loss 3.4122 | acc 0.1455 | 누적 0.5분
Epoch 2/30 | loss 2.7816 | acc 0.3055 | 누적 1.0분
Epoch 3/30 | loss 2.4342 | acc 0.4198 | 누적 1.5분
Epoch 4/30 | loss 2.1821 | acc 0.5024 | 누적 2.1분
Epoch 5/30 | loss 1.9878 | acc 0.5721 | 누적 2.6분
Epoch 6/30 | loss 1.8550 | acc 0.6147 | 누적 3.1분
Epoch 7/30 | loss 1.7383 | acc 0.6592 | 누적 3.7분
Epoch 8/30 | loss 1.6300 | acc 0.7014 | 누적 4.2분
Epoch 9/30 | loss 1.5642 | acc 0.7228 | 누적 4.8분
Epoch 10/30 | loss 1.4827 | acc 0.7534 | 누적 5.3분
Epoch 11/30 | loss 1.4367 | acc 0.7684 | 누적 5.8분
Epoch 12/30 | loss 1.3826 | acc 0.7924 | 누적 6.4분
Epoch 13/30 | loss 1.3418 | acc 0.8052 | 누적 6.9분
Epoch 14/30 | loss 1.2978 | acc 0.8205 | 누적 7.4분
Epoch 15/30 | loss 1.2573 | acc 0.8417 | 누적 8.0분
Epoch 16/30 | loss 1.2399 | acc 0.8429 | 누적 8.5분
Epoch 17/30 | loss 1.2112 | acc 0.8577 | 누적 9.1분
Epoch 18/30 | loss 1.1803 | acc 0.8710 | 누적 9.6분
Epoch 19/30 | loss 1.1557 | acc 0.8793 | 누적 10.2분
Epoch 20/30 | loss 1.1491 | acc 0.8784 | 누적 10.7분
Epoch 21/30 | loss 1.1376 |

## 6. 테스트셋 평가 + 결과 저장

### 평가 방식
- `model.eval()` + `torch.no_grad()`: 평가 모드(BatchNorm 고정, Dropout 끔) + 기울기 계산 생략
- **Overall Accuracy**: 전체 정답 비율 — 논문(ATRBench)과 동일 지표 (ResNet18 기준치 71.2%)
- **클래스별 정확도**: 어떤 차종이 어려운지 분석용

### 결과 json의 용도
예측값(`y_true`/`y_pred`) 전체를 저장 → 나중에 비교 노트북에서 **재학습/재추론 없이**
json 5개만 읽어 비교표·confusion matrix·클래스별 그래프를 생성.
**5개 모델 모두 같은 형식으로 저장해야** 비교 노트북이 동작한다.

### 가중치 2종 저장
- `{MODEL_NAME}_ckpt.pth`: 학습 재개용 (옵티마이저 포함, 큼)
- `{MODEL_NAME}_final.pth`: 추론용 모델 가중치만 — **pipeline_inference가 이걸 로드**

In [ ]:
import json
import numpy as np

model.eval()                               # 평가 모드
y_true, y_pred = [], []
with torch.no_grad():                      # 기울기 계산 끔 (속도/메모리 절약)
    for x, y in test_loader:
        out = model(x.to(device, non_blocking=True))
        y_pred.extend(out.argmax(1).cpu().tolist())   # 확률 최대 클래스 = 예측
        y_true.extend(y.tolist())

y_true, y_pred = np.array(y_true), np.array(y_pred)
overall_acc = float((y_true == y_pred).mean())

# 클래스별 정확도
per_class_acc = {cls: float((y_pred[y_true == i] == i).mean())
                 for i, cls in enumerate(test_ds.classes) if (y_true == i).sum() > 0}

print(f"Overall Accuracy: {overall_acc*100:.2f}%   (논문 ResNet18 SOC-50 기준: 71.2%)")

# ----- 비교 노트북용 결과 json (5개 모델 공통 형식) -----
result = {"model": MODEL_NAME, "setting": "SOC-50", "epochs": EPOCHS, "lr": LR,
          "overall_acc": overall_acc, "per_class_acc": per_class_acc,
          "classes": test_ds.classes,          # 클래스 순서 기록 (라벨 매핑용)
          "y_true": y_true.tolist(), "y_pred": y_pred.tolist(),   # confusion matrix용
          "history": history}                  # 학습 곡선
with open(f"{SAVE_DIR}/results/{MODEL_NAME}.json", "w") as f:
    json.dump(result, f)

# 추론용 최종 가중치 (pipeline_inference가 로드)
torch.save(model.state_dict(), f"{SAVE_DIR}/checkpoints/{MODEL_NAME}_final.pth")
print("저장 완료:", f"results/{MODEL_NAME}.json + checkpoints/{MODEL_NAME}_final.pth")

Overall Accuracy: 80.49%   (논문 ResNet18 SOC-50 기준: 71.2%)
저장 완료: results/resnet18_soc50.json + checkpoints/resnet18_soc50_final.pth


## 다음 단계

1. 드라이브에 `results/resnet18_soc50.json`과 `checkpoints/resnet18_soc50_final.pth` 생성 확인
2. 이 노트북을 복제해 셀 4(모델 정의)만 교체 → ResNet34, ConvNeXt, HDANet, SARATR-X 학습
3. 분류기가 1개라도 준비되면 `pipeline/soc50/single_resnet18.ipynb`로 탐지+분류 연결 시연 가능